# Real-data Cluster Hybrid Vecchia (3x3 target blocks)

This notebook fits the new `ClusterHybridVecchiaFit` model on July real data.

Design:
- clusters are built on regular `Latitude/Longitude` grid cells, default `3x3`;
- cluster centroids get a fresh max-min ordering inside the model;
- covariance uses the coordinates in `input_map`, so `keep_ori=True` means `Source_Latitude/Source_Longitude`;
- conditioning default is A=6 blocks at `t`, B=same+local1+shift1 at `t-1`, C=same+shift1 at `t-2`.

Start with one day. For full July, set `DAYS_LIST = list(range(31))`.

In [9]:
import sys, time, gc, contextlib, io
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import Parameter

sys.path.insert(0, '/Users/joonwonlee/Documents/GEMS_TCO-1/src')

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_cluster_hybrid import ClusterHybridVecchiaFit

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

ROUND_DECIMALS = 4

def round_numeric_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

def round_numeric_series(s, digits=ROUND_DECIMALS):
    out = s.copy()
    for idx, val in out.items():
        if isinstance(val, (float, np.floating)) and np.isfinite(val):
            out[idx] = round(float(val), digits)
    return out


torch: 2.5.1
cuda available: False


In [10]:
YEAR = '2024'
MONTH = 7
DAYS_LIST = [13]  # 0-based day index. July 14 here. Full July: list(range(31))
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
LAT_LON_RESOLUTION = [1, 1]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.double
SMOOTH = 0.5
DAILY_STRIDE = 2

# Cluster Vecchia default: conservative 3x3 block experiment.
BLOCK_SHAPE = (3, 3)
A_NEIGHBOR_BLOCKS = 6
LAG1_SAME_BLOCK = True
LAG1_LOCAL_BLOCKS = 1
LAG1_SHIFTED_BLOCKS = 1
LAG2_SAME_BLOCK = True
LAG2_LOCAL_BLOCKS = 0
LAG2_SHIFTED_BLOCKS = 1
LAG1_LON_OFFSET = 0.063
TARGET_CHUNK_SIZE = 96  # reduce to 32/64 if GPU memory is tight

LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 5
LBFGS_MAX_EVAL = 20
LBFGS_HISTORY_SIZE = 10
GRAD_TOL = 1e-5
SUPPRESS_FIT_PRINTS = False

INIT_PHYSICAL = {
    'sigmasq': 10.0,
    'range_lat': 0.30,
    'range_lon': 0.40,
    'range_time': 2.0,
    'advec_lat': 0.05,
    'advec_lon': -0.10,
    'nugget': 2.5,
}

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log')
OUT_PREFIX = 'real_vecc_2024_cluster_hybrid_051326'
SAVE_CSV = True

print('device:', DEVICE)
print('days:', DAYS_LIST)
print('block:', BLOCK_SHAPE)
print('conditioning blocks:', {
    'A': A_NEIGHBOR_BLOCKS,
    'B': f'same{int(LAG1_SAME_BLOCK)}+local{LAG1_LOCAL_BLOCKS}+shift{LAG1_SHIFTED_BLOCKS}',
    'C': f'same{int(LAG2_SAME_BLOCK)}+local{LAG2_LOCAL_BLOCKS}+shift{LAG2_SHIFTED_BLOCKS}',
})

device: cpu
days: [13]
block: (3, 3)
conditioning blocks: {'A': 6, 'B': 'same1+local1+shift1', 'C': 'same1+local0+shift1'}


## Load Real Data

Important: this model does **not** need point-level max-min ordering or point `nns_map`. We load the monthly data with `is_whittle=True` to avoid generating the old point neighbor map. The cluster model builds its own cluster max-min ordering internally.

In [11]:
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

df_map, _, _, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=LAT_LON_RESOLUTION,
    mm_cond_number=1,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)

key_idx = sorted(df_map)
print('n hourly slots:', len(key_idx))
print('monthly_mean:', monthly_mean)
print('first key:', key_idx[0], 'last key:', key_idx[-1])

base_grid_coords_np = df_map[key_idx[0]][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
print('base_grid_coords_np:', base_grid_coords_np.shape)
print('n unique lat/lon:', len(np.unique(base_grid_coords_np[:,0])), len(np.unique(base_grid_coords_np[:,1])))

--- Global Monthly Mean for 2024-7: 257.9726 ---
n hourly slots: 248
monthly_mean: 257.9726104252314
first key: 2024_07_y24m07day01_hm00:53 last key: 2024_07_y24m07day31_hm07:48
base_grid_coords_np: (18126, 2)
n unique lat/lon: 114 159


In [12]:
def assert_grid_order_consistent(keys, base_coords):
    for k in keys:
        coords = df_map[k][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
        if coords.shape != base_coords.shape or not np.allclose(coords, base_coords, equal_nan=True):
            raise RuntimeError(f'Grid coordinate order differs at {k}; cluster local-index mapping is not reusable.')

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    assert_grid_order_consistent(key_idx[hour_indices[0]:hour_indices[1]], base_grid_coords_np)
print('grid order consistency check passed for selected days')

grid order consistency check passed for selected days


In [13]:
daily_hourly_maps = {}
selected_key_map = {}

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    selected_key_map[day_idx] = key_idx[hour_indices[0]:hour_indices[1]]
    day_hourly_map, _ = data_load_instance.load_working_data(
        df_map,
        monthly_mean,
        hour_indices,
        ord_mm=None,       # no point max-min ordering; cluster model handles cluster ordering
        dtype=DTYPE,
        keep_ori=True,     # covariance uses Source_Latitude/Source_Longitude
    )
    daily_hourly_maps[day_idx] = {k: v.to(DEVICE) for k, v in day_hourly_map.items()}

rows = []
for day_idx, maps in daily_hourly_maps.items():
    n_valid = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in maps.values())
    n_total = sum(int(v.shape[0]) for v in maps.values())
    rows.append({
        'day_idx': day_idx,
        'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
        'n_time_slots': len(maps),
        'n_rows_total': n_total,
        'n_valid_o3': n_valid,
        'valid_rate': n_valid / n_total,
        'first_slot': selected_key_map[day_idx][0],
        'last_slot': selected_key_map[day_idx][-1],
    })
load_summary = pd.DataFrame(rows)
display(round_numeric_df(load_summary))

,day_idx,day,n_time_slots,n_rows_total,n_valid_o3,valid_rate,first_slot,last_slot
0,13,2024-07-14,8,145008,144078,0.9936,2024_07_y24m07day14_hm00:53,2024_07_y24m07day14_hm07:48


## Parameter Helpers

In [14]:
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

def physical_to_log_phi(params):
    sigmasq = float(params['sigmasq'])
    range_lat = float(params['range_lat'])
    range_lon = float(params['range_lon'])
    range_time = float(params['range_time'])
    nugget = float(params['nugget'])
    phi2 = 1.0 / range_lon
    phi1 = sigmasq * phi2
    phi3 = (range_lon / range_lat) ** 2
    phi4 = (range_lon / range_time) ** 2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        float(params['advec_lat']), float(params['advec_lon']), np.log(nugget),
    ]

def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq': np.exp(p[0]) / phi2,
        'range_lat': rlon / phi3 ** 0.5,
        'range_lon': rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat': p[4],
        'advec_lon': p[5],
        'nugget': np.exp(p[6]),
    }

def make_params_list(init_physical=INIT_PHYSICAL):
    initial_vals = physical_to_log_phi(init_physical)
    return [Parameter(torch.tensor([val], dtype=DTYPE, device=DEVICE)) for val in initial_vals]

print('init log phi:', np.round(physical_to_log_phi(INIT_PHYSICAL), 4))

init log phi: [ 3.2189  0.9163  0.5754 -3.2189  0.05   -0.1     0.9163]


## Fit One Day

In [15]:
def instantiate_model(day_map):
    return ClusterHybridVecchiaFit(
        smooth=SMOOTH,
        input_map=day_map,
        grid_coords=base_grid_coords_np,
        block_shape=BLOCK_SHAPE,
        n_neighbor_blocks_t=A_NEIGHBOR_BLOCKS,
        lag1_same_block=LAG1_SAME_BLOCK,
        lag1_local_blocks=LAG1_LOCAL_BLOCKS,
        lag1_shifted_blocks=LAG1_SHIFTED_BLOCKS,
        lag2_same_block=LAG2_SAME_BLOCK,
        lag2_local_blocks=LAG2_LOCAL_BLOCKS,
        lag2_shifted_blocks=LAG2_SHIFTED_BLOCKS,
        daily_stride=DAILY_STRIDE,
        lag1_lon_offset=LAG1_LON_OFFSET,
        target_chunk_size=TARGET_CHUNK_SIZE,
    )

def fit_one(day_idx):
    day_map = daily_hourly_maps[day_idx]
    params_list = make_params_list()
    model = instantiate_model(day_map)

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(
        params_list,
        lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL,
        max_eval=LBFGS_MAX_EVAL,
        history_size=LBFGS_HISTORY_SIZE,
    )

    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    else:
        out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    fit_s = time.time() - t1

    est = backmap_params(out)
    row = {
        'day_idx': day_idx,
        'day': f'{YEAR}-{MONTH:02d}-{day_idx + 1:02d}',
        'smooth': SMOOTH,
        'block_shape': f'{BLOCK_SHAPE[0]}x{BLOCK_SHAPE[1]}',
        'loss': float(out[-1]),
        'steps_raw': int(steps_ran),
        'precompute_s': round(precompute_s, ROUND_DECIMALS),
        'fit_s': round(fit_s, ROUND_DECIMALS),
        **{f'est_{k}': float(est[k]) for k in P_LABELS},
        **model.cluster_summary(),
    }

    del model, params_list, optimizer
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return row

results = []
for day_idx in DAYS_LIST:
    print(f'\n=== fitting day_idx={day_idx} ({YEAR}-{MONTH:02d}-{day_idx+1:02d}) ===')
    row = fit_one(day_idx)
    results.append(row)
    print(round_numeric_series(pd.Series(row)).to_string())

results_df = pd.DataFrame(results)
display(round_numeric_df(results_df))


=== fitting day_idx=13 (2024-07-14) ===
Pre-computing ClusterHybridVecchia (smooth=0.5, block=(3, 3), A=6, B=same1+local1+fresh1, C=same1+local0+fresh1)... Done. clusters=2014, max_points/block=9, target_blocks=16110, target_points=144078, batches=[A:m54:b6x19, A:m54:b8x1, A:m54:b9x1994, AB:m81:b4x1, AB:m81:b5x1, AB:m81:b6x15, AB:m81:b8x3, AB:m81:b9x1994, ABC:m99:b1x4, ABC:m99:b2x5, ... (17 batches)]
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/5 / Loss: 1.109601 ---
  Param 0: Value=3.3262, Grad=-3.835979754315557e-05
  Param 1: Value=1.2661, Grad=3.065198252119205e-05
  Param 2: Value=0.4000, Grad=2.7955145628895887e-05
  Param 3: Value=-3.3424, Grad=4.296383618521755e-05
  Param 4: Value=0.0078, Grad=-9.489067488893796e-05
  Param 5: Value=-0.0286, Grad=1.5076965318527088e-05
  Param 6: Value=0.1107, Grad=-3.1510599439047324e-05
  Max Abs Grad: 9.489067e-05
------------------------------
--- Step 2/5 / Loss: 1.069682 ---
  Param 0: Value=3.3269, Grad=1.184221429396

,day_idx,day,smooth,block_shape,loss,steps_raw,precompute_s,fit_s,est_sigmasq,est_range_lat,...,est_advec_lon,est_nugget,n_clusters,block_shape_lat,block_shape_lon,max_points_per_cluster,n_target_blocks,n_target_points,n_batches,target_chunk_size
0,13,2024-07-14,0.5,3x3,1.0697,4,0.5923,118.8527,7.87,0.2316,...,-0.0288,1.1172,2014,3,3,9,16110,144078,17,96


In [16]:
if SAVE_CSV:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUT_DIR / f'{OUT_PREFIX}_days_{min(DAYS_LIST)+1:02d}_{max(DAYS_LIST)+1:02d}.csv'
    round_numeric_df(results_df).to_csv(out_path, index=False, float_format="%.4f")
    print('saved:', out_path)

saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_cluster_hybrid_051326_days_14_14.csv
